<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session4/1-Text-classification-BERT.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 4 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificación de textos - BERT

En este notebook se aborda el problema de clasificación de texto aplicado a la detección de contenido relacionado con el cambio climático. Para ello, se utiliza el dataset somosnlp/spa_climate_detection, el cual contiene textos en español etiquetados según su relación con esta temática.

El objetivo principal es desarrollar un modelo capaz de clasificar automáticamente textos en función de si están relacionados o no con el cambio climático. Para lograrlo, se emplea el modelo BERT, una arquitectura basada en transformers que permite capturar el contexto semántico del lenguaje de manera bidireccional, mejorando significativamente el rendimiento en tareas de procesamiento de lenguaje natural (NLP).

# 1. Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las reseñas.

In [ ]:
import importlib.metadata
import warnings

warnings.filterwarnings('ignore')

installed_packages = [dist.metadata['Name'].lower() for dist in importlib.metadata.distributions()]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
!pip install torchinfo #> /dev/null 2>&1
!pip install evaluate #> /dev/null 2>&1
!pip install transformers datasets

# 2. Recopilación de datos

Para el presente análisis se utilizará el conjunto de datos somosnlp/spa_climate_detection, disponible en Hugging Face, el cual contiene textos en español relacionados con la temática del cambio climático. Cada registro del dataset se encuentra etiquetado según si el contenido está o no asociado a este tema.

Este conjunto de datos permitirá evaluar el desempeño de modelos de procesamiento de lenguaje natural en una tarea de clasificación binaria de texto. En particular, se empleará el modelo BERT, el cual será ajustado (fine-tuning) sobre este corpus para aprovechar su capacidad de comprensión contextual del lenguaje.

De esta manera, se busca construir un modelo capaz de identificar automáticamente si un texto está relacionado con el cambio climático, evaluando su rendimiento mediante métricas estándar de clasificación.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('somosnlp/spa_climate_detection', split='train')
dataset

In [ ]:
dataset[1]

# 3. Análisis Exploratorio

En esta sección se realiza un análisis exploratorio del conjunto de datos somosnlp/spa_climate_detection con el objetivo de comprender su estructura, distribución y características principales antes del proceso de modelado.

Inicialmente, se examina la cantidad total de registros, así como la distribución de las clases, con el fin de identificar posibles problemas de desbalanceo que puedan afectar el desempeño del modelo. Asimismo, se analizan características básicas de los textos, como la longitud de las secuencias, la frecuencia de palabras y la presencia de términos relevantes asociados al cambio climático.

Adicionalmente, se exploran ejemplos representativos de cada clase para entender mejor el tipo de contenido presente en el dataset y la complejidad de la tarea de clasificación. Este análisis permite identificar patrones, posibles ruidos en los datos y definir estrategias adecuadas de preprocesamiento y modelado.



## Estructura del Dataset

In [ ]:
df = dataset.to_pandas()
df.head()

Revisamos los primero 5 registros del dataset para validar estructura y valores

## Análisis de Distribución

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

labels = dataset["answer"]

# Contar clases
counter = Counter(labels)

# Mostrar conteo
print("Distribución absoluta:")
print(counter)

# Calcular proporciones
total = sum(counter.values())
print("\nDistribución porcentual:")
for cls, freq in counter.items():
    print(f"Clase {cls}: {freq} ({freq/total:.2%})")


plt.figure()
bars = plt.bar(counter.keys(), counter.values())

plt.xticks(list(counter.keys()))
plt.xlabel("Clase (0 = No clima, 1 = Clima)")
plt.ylabel("Número de registros")
plt.title("Distribución de clases en el dataset")

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f'{height}', ha='center', va='bottom')

plt.show()

La distribución de clases en el dataset es relativamente equilibrada, con una ligera predominancia de la clase 1 (Texto relacionado con cambio climatico) (55.17%) frente a la clase 0 (Texto no relacionado con cambio climático) (44.83%).

Esto sugiere que no existe un desbalance significativo que pueda afectar gravemente el entrenamiento del modelo.


## Análisis del Texto

In [ ]:
import numpy as np

text_lengths = [len(row['question'].split()) for row in dataset]

print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"{p}%: {np.percentile(text_lengths, p)}")

Los textos del dataset presentan una alta variabilidad en su longitud, con valores que van desde 2 hasta 604 palabras.

La longitud promedio es de aproximadamente 114 palabras, aunque la mediana (50%) se sitúa en 59, lo que indica una distribución sesgada con presencia de textos considerablemente más largos.

 Esto se confirma con los percentiles superiores, donde un pequeño porcentaje de textos alcanza longitudes significativamente mayores.

In [ ]:
df['Palabras por clase'] = df['question'].str.split().apply(len)
df.groupby('answer')['Palabras por clase'].median()

In [ ]:
df.boxplot(column='Palabras por clase', by='answer', grid=False)
plt.title('Distribución de longitud de textos por clase')
plt.suptitle('')
plt.xlabel('Clase (0 = No clima, 1 = Clima)')
plt.ylabel('Número de palabras')
plt.show()

Se evidencia diferencias en la distribución de la longitud de los textos entre las clases. La clase 0 presenta una mayor dispersión y valores más extremos, indicando la presencia de textos considerablemente más largos, mientras que la clase 1 tiene una distribución más concentrada y homogénea.

Esto sugiere que la longitud del texto podría variar según la clase, aunque con cierto solapamiento entre ambas.

## Vocabulario

In [ ]:
from collections import Counter

all_words = []
for text in dataset["question"]:
    all_words.extend(text.split())

vocab = Counter(all_words)
print("Tamaño del vocabulario:", len(vocab))

El dataset cuenta con 41136 palabras únicas lo que indica una alta diversidad léxica en los textos.

## Análisis de Ruido

In [ ]:
df.sample(5)

Podemos ver aleatoriamente algunos registros para rectificar la clasificación o valor de la etiqueta vs el texto

## Palabras más frecuentes y N-Gramas por clase

Analizamos las palabras y bigramas más frecuentes en cada polaridad, excluyendo stopwords, para identificar el vocabulario diferenciador entre comentarios de cada clase.

In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("spanish"))
extra_stopwords = {
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'del', 'se', '(', ')',
    'que', 'con', 'un', 'una', 'es', 'no', 'lo', 'su', 'por', 'al', ',', '.', 'más', 'le', 'me', 'mi'
}
stop_words.update(extra_stopwords)

def top_words_by_class(ds, label, n=15, sw=None):
    words = []
    for row in ds:
        if row['answer'] == label:
            words.extend(row['question'].lower().split())
    if sw:
        words = [w for w in words if w not in sw]
    return Counter(words).most_common(n)

def get_bigrams(text):
    tokens = text.lower().split()
    return [f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens) - 1)]

top_pos = top_words_by_class(dataset, 1, sw=stop_words)
top_neg = top_words_by_class(dataset, 0, sw=stop_words)

bg_pos, bg_neg = Counter(), Counter()
for row in dataset:
    bgs = get_bigrams(row['question'])
    if row['answer'] == 1:
        bg_pos.update(bgs)
    else:
        bg_neg.update(bgs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, top, title, color in [
    (axes[0, 0], top_pos,                'Top 15 palabras — Relacionado', 'steelblue'),
    (axes[0, 1], top_neg,                'Top 15 palabras — No relacionado',  'salmon'),
    (axes[1, 0], bg_pos.most_common(10), 'Top 10 bigramas — Relacionado', 'steelblue'),
    (axes[1, 1], bg_neg.most_common(10), 'Top 10 bigramas — No relacionado',  'salmon'),
]:
    terms, freqs = zip(*top)
    ax.barh(list(terms)[::-1], list(freqs)[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

- **Existe una clara diferenciación léxica entre clases**: en la clase relacionada con cambio climático predominan términos como *cambio, climático, emisiones, calentamiento y carbono*, lo que indica un vocabulario altamente específico del dominio.

- **Clase relacionada (1)** : se observan palabras directamente asociadas al contexto ambiental y científico, lo que sugiere que los textos contienen información más técnica o explícitamente vinculada al cambio climático.

- **Clase no relacionada (0)**: predominan términos más generales o contextuales como *gobierno, madrid, partido, vox*, lo que indica que muchos textos pertenecen a ámbitos políticos o sociales, pero no necesariamente al tema climático.

- **Baja superposición semántica relevante**: a diferencia de otros problemas (como sentimiento), aquí las clases parecen diferenciarse más por el tema que por el tono, lo que facilita la tarea de clasificación.

## Sobre los n-gramas (bigramas)

- Los bigramas más frecuentes (ej. de la, en el, de los) son principalmente estructuras gramaticales comunes del español, por lo que aportan poco valor discriminativo por sí solos.

- En la clase relacionada, aparecen combinaciones más informativas como *cambio climático, millones de, el cambio*, que refuerzan el contexto temático.

- En la clase no relacionada, los bigramas siguen siendo mayormente genéricos y estructurales, lo que confirma la menor presencia de expresiones específicas del dominio climático.

- Para mejorar la capacidad discriminativa, sería útil enfocarse en n-gramas más semánticos o aplicar técnicas que capturen mejor el contexto, como embeddings generados por BERT.


# 4. Tokenizador

En esta etapa se realiza el proceso de tokenización del texto utilizando el tokenizador preentrenado asociado al modelo BERT. En particular, se emplea el tokenizador correspondiente al modelo dccuchile/bert-base-spanish-wwm-cased, disponible en la librería Hugging Face Transformers.

Es importante destacar que, para modelos basados en transformers, se debe utilizar el mismo tokenizador con el que el modelo fue entrenado originalmente. Esto garantiza que la representación de los textos sea consistente con la arquitectura del modelo y permite aprovechar adecuadamente el conocimiento previamente aprendido durante el preentrenamiento.

El tokenizador de BERT divide el texto en sub-palabras (subwords), lo que permite manejar palabras desconocidas y capturar mejor la estructura del lenguaje. Como resultado, cada texto es transformado en una secuencia de identificadores numéricos (tokens), que pueden ser procesados posteriormente por el modelo para la tarea de clasificación.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tokenizers.pre_tokenizers import ByteLevel

model_name = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

Probamos el tokenizador obtenido

In [ ]:
tokenizer.pad_token = '[PAD]'
tokenizer("hola mundo!!", max_length=10, truncation=True, padding='max_length').tokens()

Tamaño del vocabulario

In [ ]:
tokenizer.vocab_size

Longitud máxima de tokens que el modelo puede procesar en una sola entrada.

In [ ]:
tokenizer.model_max_length

Revisamos los nombres de las entradas que el modelo espera recibir después del proceso de tokenización

In [ ]:
tokenizer.model_input_names

¿Qué significa cada uno?

- input_ids:
Son los tokens convertidos a números (lo que realmente procesa el modelo)

- attention_mask:
Indica qué tokens son reales (1) y cuáles son padding (0)

- token_type_ids:
Se usan para diferenciar segmentos de texto (por ejemplo, en tareas con dos oraciones)

In [ ]:
tokens = sorted(tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[:15]])
print("15 tokens de en medio:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[20000:20020]])
print("Últimos 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[-15:]])

# 5. Definición del modelo BERT pre-entrenado

En esta etapa se define el modelo preentrenado que será utilizado para la tarea de clasificación de texto. Se emplea el modelo dccuchile/bert-base-spanish-wwm-cased, basado en la arquitectura BERT, el cual ha sido previamente entrenado sobre grandes volúmenes de texto en español.

El uso de un modelo preentrenado permite aprovechar el conocimiento lingüístico adquirido durante su entrenamiento inicial, facilitando la comprensión del contexto y las relaciones semánticas entre palabras. Posteriormente, este modelo es ajustado (fine-tuning) específicamente para la tarea de clasificación de textos relacionados con el cambio climático.

De esta manera, se logra una mejora significativa en el rendimiento del modelo en comparación con enfoques tradicionales, al utilizar representaciones profundas del lenguaje.

In [ ]:
id2category = dict(enumerate(np.unique(df['answer'])))
category2id = {v: k for k, v in id2category.items()}

In [ ]:
import torch
from torchinfo import summary
from transformers import AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer("hola mundo!!!", max_length=10, truncation=True, padding='max_length', return_tensors='pt')

print(f"Input Shapes & Types:")
print({k: (v.shape, v.dtype) for k, v in inputs.items()})

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

# Congelamos los pesos del modelo base para usarlo como featurizer solamente.
for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Se utiliza BERT como extractor de características, manteniendo sus pesos congelados y entrenando únicamente la capa de clasificación final

In [ ]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
print({k: v.shape for k, v in outputs.items()})

La salida del modelo contiene los logits con dimensión [1, 2], donde cada valor representa la puntuación asociada a cada clase en la clasificación binaria

In [ ]:
outputs

In [ ]:
model.classifier

Esta capa lineal transforma la representación generada por BERT (768 dimensiones) en dos valores, correspondientes a las clases del problema de clasificación binaria

### Instanciamos los datasets

Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y prueba (10%).

In [ ]:
training_dataset = dataset.train_test_split(train_size=0.8)
validation_dataset = training_dataset['test'].train_test_split(train_size=0.5)

In [ ]:
from datasets.dataset_dict import DatasetDict

new_dataset = DatasetDict({
    'train': training_dataset['train'],
    'val': validation_dataset['train'],
    'test': validation_dataset['test'],
})
new_dataset

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['question'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

def tokenize(max_len: int = 8):
    def _tokenize(batch):
        return tokenizer(batch['question'], max_length=max_len, truncation=True, padding='max_length')
    return _tokenize

def category_names_2_ids(batch):
    batch['label'] = category2id[batch['answer']]
    return batch

Se tokenizan los textos y se convierten las etiquetas a formato numérico, generando un dataset listo para ser utilizado en el entrenamiento del modelo.

In [ ]:
tokenized_dataset = new_dataset.map(preprocess_function(max_len=512), batched=True)
tokenized_dataset = tokenized_dataset.map(category_names_2_ids)
tokenized_dataset

### Definición del proceso de entrenamiento

En esta etapa se configuran los parámetros y el proceso de entrenamiento del modelo utilizando la clase Trainer de la librería Hugging Face Transformers. Este componente facilita la gestión del entrenamiento, evaluación y registro de métricas de manera estructurada.

In [ ]:
from transformers import Trainer, TrainingArguments
from typing import Dict, Any
import evaluate

# Definimos la función métrica de calidad
accuracy = evaluate.load("accuracy")

def compute_metrics(pred) -> Dict[str, Any]:
    """compute metrics

    Esta función será invocada en
    cada epoca y la utilizaremos para
    calcular la métrica de calidad.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Retorna un diccionario como {'nombre-metrica': valor}
    acc = accuracy.compute(predictions=preds, references=labels)
    return acc


batch_size = 8 if IN_COLAB else 4
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./bert_preentrenado',
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='tensorboard'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

# Métricas detalladas de clasificación

En adelante, para cada modelo evaluaremos:

- Reporte de clasificación (precision, recall, F1 por clase)

- Matriz de confusión

Esto permite analizar mejor el sesgo entre clases y no depender solo de la accuracy global.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


def evaluate_with_reports(trainer, dataset_split, split_name='test', label_names=None):
    pred_output = trainer.predict(dataset_split)
    y_true = pred_output.label_ids
    y_pred = pred_output.predictions.argmax(-1)

    if label_names is None:
        unique_labels = sorted(set(y_true))
        label_names = [str(label) for label in unique_labels]

    print(f"\n=== Reporte de clasificación ({split_name}) ===")
    print(classification_report(y_true, y_pred, target_names=label_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(label_names))))
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"Matriz de confusión ({split_name})")
    plt.show()

    return y_true, y_pred, pred_output

Ejecutamos el entrenamiento

In [ ]:
%%time
trainer.train()

Revisamos metricas de rendimiento

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir bert_preentrenado/runs

**Resultados del entrenamiento**


- La pérdida de entrenamiento disminuye de 0.678 a 0.642, lo que indica que el modelo está aprendiendo patrones en los datos.

- La pérdida de validación también presenta una ligera reducción, sugiriendo que el modelo generaliza de manera estable.

- La accuracy se mantiene constante en 0.662, lo que indica que, aunque el modelo mejora en términos de pérdida, no logra incrementar su capacidad de clasificación en el conjunto de validación.

Evaluamos en el conjunto de prueba

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
label_names = ['No clima', 'Clima']

y_true_base, y_pred_base, pred_base = evaluate_with_reports(
    trainer=trainer,
    dataset_split=tokenized_dataset['test'],
    split_name='BERT base (congelado)',
    label_names=label_names
)

**Evaluacion en el conjunto de prueba**

- La correctitud (accuracy) de 70.3% indica que el modelo tiene una capacidad aceptable para clasificar correctamente los textos, superando el desempeño observado durante la validación (~66%).

- La reducción en la pérdida (loss) frente a validación sugiere que el modelo logra una mejor generalización sobre datos no vistos.

- El rendimiento obtenido es consistente con un modelo que ha capturado patrones relevantes del lenguaje, aunque aún existe margen de mejora.

In [ ]:
predictions = trainer.predict(tokenized_dataset['test'])

In [ ]:
predicted_labels = np.argmax(predictions.predictions, axis=-1)
test_set = tokenized_dataset['test']
test_set = test_set.add_column('prediction_label', predicted_labels)
test_set = test_set.add_column('prediction', list(map(lambda label: id2category[label], predicted_labels)))
test_set

Revisamos las etiquetas reales vs las predichas

In [ ]:
columns = ['question', 'answer',  'prediction']
test_set.set_format('pandas', columns=columns)
df = test_set.to_pandas()[columns]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

Revisamos dónde el modelo no acertó en la predicción

In [ ]:
errors = df[df['answer'] != df['prediction']]
errors.head(15)

# 5.1. Definición del modelo BERTIN ROBERTA

Ahora emplearemos el modelo bertin-project/bertin-roberta-base-spanish, basado en la arquitectura RoBERTa, el cual ha sido entrenado previamente sobre grandes volúmenes de texto en español.

A diferencia de los modelos basados en BERT tradicionales, RoBERTa incorpora mejoras en el proceso de entrenamiento, como una optimización en el uso de datos, eliminación de ciertas restricciones en el preentrenamiento y un ajuste más robusto de los hiperparámetros, lo que permite obtener representaciones lingüísticas más ricas y precisas.

Usar mismo tokenizador con el que fue entrenado el modelo

In [ ]:
model_bertin = "bertin-project/bertin-roberta-base-spanish"

tokenizer_bertin = AutoTokenizer.from_pretrained(model_bertin)

In [ ]:
tokenizer_bertin.pad_token = '[PAD]'
tokenizer_bertin("hola mundo!!", max_length=10, truncation=True, padding='max_length').tokens()

Podemos observar una diferencia en la tokenizacion

In [ ]:
tokenizer_bertin.vocab_size

A diferencia de modelos como dccuchile/bert-base-spanish-wwm-cased, que cuentan con un vocabulario cercano a 30.000 tokens, bertin-project/bertin-roberta-base-spanish utiliza un vocabulario más amplio de 50.262 tokens, lo que permite una mejor representación del lenguaje al reducir la fragmentación en subpalabras, aunque con un ligero incremento en el costo computacional.

In [ ]:
tokenizer_bertin.model_max_length

In [ ]:
tokenizer_bertin.model_input_names

In [ ]:
tokens_bertin = sorted(tokenizer_bertin.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{tokenizer_bertin.convert_tokens_to_string([t])}" for t, _ in tokens_bertin[:15]])
print("15 tokens de en medio:")
print([f"{tokenizer_bertin.convert_tokens_to_string([t])}" for t, _ in tokens_bertin[20000:20020]])
print("Últimos 15 tokens:")
print([f"{tokenizer_bertin.convert_tokens_to_string([t])}" for t, _ in tokens_bertin[-15:]])

Definimos el modelo BERTIN pre entrenado

In [ ]:
model_bertin = AutoModelForSequenceClassification.from_pretrained(model_bertin, num_labels=len(category2id)).to(device)

# Congelamos los pesos del modelo base para usarlo como featurizer solamente.
for param in model_bertin.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model_bertin, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model_bertin(**inputs)
print({k: v.shape for k, v in outputs.items()})

In [ ]:
outputs

In [ ]:
model_bertin.classifier

Creamos los datasets con el nuevo tokenizador

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer_bertin(examples['question'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

def tokenize(max_len: int = 8):
    def _tokenize(batch):
        return tokenizer_bertin(batch['question'], max_length=max_len, truncation=True, padding='max_length')
    return _tokenize

def category_names_2_ids(batch):
    batch['label'] = category2id[batch['answer']]
    return batch

In [ ]:
tokenized_bertin_dataset = new_dataset.map(preprocess_function(max_len=512), batched=True)
tokenized_bertin_dataset = tokenized_bertin_dataset.map(category_names_2_ids)
tokenized_bertin_dataset

Definición del proceso de entrenamiento

In [ ]:
batch_size = 8 if IN_COLAB else 4
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args_bertin = TrainingArguments(
    output_dir='./bertin_preentrenado',
    num_train_epochs=2,
    learning_rate=2e-5,
    lr_scheduler_type='linear',
    optim="adamw_torch",
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='tensorboard'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer_bertin = Trainer(
    model=model_bertin,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_bertin_dataset['train'],
    eval_dataset=tokenized_bertin_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer_bertin.train()

In [ ]:
model_bertin.eval()
trainer_bertin.evaluate(tokenized_bertin_dataset['test'])

In [ ]:
y_true_bertin, y_pred_bertin, pred_bertin= evaluate_with_reports(
    trainer=trainer_bertin,
    dataset_split=tokenized_bertin_dataset['test'],
    split_name='BERTIN ROBERTA base (congelado)',
    label_names=label_names
)

En comparación con el modelo dccuchile/bert-base-spanish-wwm-cased, el modelo bertin-project/bertin-roberta-base-spanish presenta un mejor desempeño general. BERTin logra una mayor precisión en ambas épocas (0.6965 vs 0.6655 en la primera, y 0.7069 vs 0.7034 en la segunda), además de mostrar pérdidas de entrenamiento y validación ligeramente más bajas.

Estos resultados indican que BERTin RoBERTa tiene una mejor capacidad de generalización y aprendizaje del contexto en el conjunto de datos, aunque la diferencia final en accuracy es moderada. En conjunto, se evidencia una ventaja consistente de BERTin sobre BERT para esta tarea específica.

# 6. Uso de clasificador especializado

Hasta este punto, se ha utilizado la configuración estándar de BertForSequenceClassification, la cual emplea una única capa lineal como clasificador final. Si bien esta aproximación es eficiente, puede resultar limitada para capturar relaciones más complejas en los datos.

Con el objetivo de mejorar el desempeño del modelo, se propone implementar un clasificador personalizado, añadiendo mayor profundidad a la arquitectura mediante múltiples capas densas. En este enfoque, el modelo BERT continúa utilizándose como extractor de características (feature extractor), mientras que la nueva cabeza de clasificación introduce mayor capacidad de aprendizaje no lineal.

Cargamos nuevamente el modelo

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Rectificamos el clasificador

In [ ]:
model.classifier

El clasificador propuesto está compuesto por:

- Capa Lineal (768 → 256): reducción de dimensionalidad y extracción de características relevantes

- Layer Normalization: mejora la estabilidad del entrenamiento

- ReLU: introduce no linealidad

- Dropout: reduce el riesgo de sobreajuste

- Capa final (256 → 2): genera los logits para la clasificación binaria

In [ ]:
import torch.nn as nn


classifier = nn.Sequential(
    nn.Linear(768, 256),
    nn.LayerNorm(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)
)

# simplemente reemplazamos el clasificador existente por el nuestro:
model.classifier = classifier
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time

trainer.train()

Tras implementar el clasificador personalizado sobre BERT, se observan mejoras significativas en el desempeño del modelo:

- La pérdida de entrenamiento disminuye de 0.53 a 0.40, evidenciando un aprendizaje efectivo.

- La pérdida de validación también se reduce de 0.40 a 0.37, lo que indica una buena capacidad de generalización.

- La accuracy alcanza un 82.76%, mostrando una mejora considerable frente al modelo base (~66%).

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
y_true_custom, y_pred_custom, pred_custom = evaluate_with_reports(
    trainer=trainer,
    dataset_split=tokenized_dataset['test'],
    split_name='BERT + clasificador personalizado',
    label_names=label_names
)

- La accuracy del 86.55% evidencia una mejora significativa respecto al modelo base, confirmando la efectividad del clasificador personalizado.

- La baja pérdida (loss) indica que el modelo no solo acierta en sus predicciones, sino que lo hace con alta confianza.

- El desempeño superior en test frente a validación (~82.7%) sugiere una buena capacidad de generalización.

# 7. Fine-tuning del modelo BERT

En esta etapa se aplica fine-tuning sobre el modelo BERT, lo que implica liberar todas las capas del modelo para que sus parámetros sean actualizados durante el entrenamiento.

A diferencia del enfoque anterior, donde el modelo se utilizaba como extractor de características con pesos congelados, en este caso se permite que toda la arquitectura aprenda de la tarea específica. Esto posibilita un ajuste más profundo de las representaciones internas, adaptándolas al dominio del problema.

Para este proceso, no es necesario modificar la estructura del modelo preentrenado; simplemente se instancia el modelo base y se entrena directamente sobre el dataset, permitiendo que todos sus parámetros participen en la optimización.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer.train()

Tras aplicar fine-tuning sobre el modelo BERT, se obtienen los siguientes resultados:

- La pérdida de entrenamiento disminuye significativamente de 0.187 a 0.062, evidenciando un aprendizaje muy efectivo.

- La pérdida de validación se mantiene en valores muy bajos (~0.07), lo que indica un excelente ajuste del modelo a los datos.

- La accuracy alcanza un 98.27%, reflejando un desempeño sobresaliente en la tarea de clasificación.

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
y_true_ft, y_pred_ft, pred_ft = evaluate_with_reports(
    trainer=trainer,
    dataset_split=tokenized_dataset['test'],
    split_name='BERT fine-tuning completo',
    label_names=label_names
)

- La accuracy cercana al 98% confirma un desempeño sobresaliente del modelo en datos no vistos.

- La baja pérdida (loss) indica que el modelo no solo predice correctamente, sino que lo hace con alta confianza.

- La consistencia entre validación (98.27%) y test (97.59%) sugiere una excelente capacidad de generalización.

# Comparación consolidada de modelos

Esta tabla resume las métricas más importantes para comparar los tres enfoques evaluados:

- Accuracy

- Balanced Accuracy

- Macro F1

- Recall por clase (No clima / Clima)

- Precision por clase (No clima / Clima)


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, precision_score
import pandas as pd

results_summary = []

model_predictions = {
    'BERT base (congelado)': (y_true_base, y_pred_base),
    'BERTIN ROBERTA base (congelado)': (y_true_bertin, y_pred_bertin),
    'BERT + clasificador personalizado': (y_true_custom, y_pred_custom),
    'BERT fine-tuning completo': (y_true_ft, y_pred_ft)
}

for model_name, (y_true, y_pred) in model_predictions.items():
    results_summary.append({
        'Modelo': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall No clima (0)': recall_score(y_true, y_pred, pos_label=0, average='binary', zero_division=0),
        'Recall Clima (1)': recall_score(y_true, y_pred, pos_label=1, average='binary', zero_division=0),
        'Precision No clima (0)': precision_score(y_true, y_pred, pos_label=0, average='binary', zero_division=0),
        'Precision Clima (1)': precision_score(y_true, y_pred, pos_label=1, average='binary', zero_division=0)
    })

results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values(by='Macro F1', ascending=False).reset_index(drop=True)

numeric_cols = [
    'Accuracy', 'Balanced Accuracy', 'Macro F1',
    'Recall No clima (0)', 'Recall Clima (1)',
    'Precision No clima (0)', 'Precision Clima (1)'
]
results_df[numeric_cols] = results_df[numeric_cols].round(4)

results_df

Los resultados muestran que el modelo dccuchile/bert-base-spanish-wwm-cased con fine-tuning completo logra el mejor desempeño global, alcanzando métricas muy altas y balanceadas en todas las clases, lo que indica una excelente capacidad de generalización.

En contraste, los modelos con BERT congelado y bertin-project/bertin-roberta-base-spanish congelado presentan un rendimiento considerablemente inferior, con un fuerte sesgo hacia la clase “clima”, evidenciado por recalls cercanos a 1.0 pero muy bajos en la clase “no clima”. Esto indica que, sin ajuste fino, los embeddings preentrenados no son suficientes para la tarea.

El modelo con clasificador personalizado mejora frente al modelo congelado, pero aún presenta desbalance entre clases, priorizando la detección de “clima” sobre “no clima”.

En conjunto, se concluye que el fine-tuning completo es fundamental para obtener un buen desempeño, y que el uso de modelos congelados limita significativamente la capacidad de discriminación del modelo.

# 8. Conclusiones

En este notebook se implementaron y evaluaron diferentes estrategias basadas en la arquitectura BERT para la tarea de clasificación de textos relacionados con el cambio climático, utilizando el dataset somosnlp/spa_climate_detection.

### Enfoques evaluados

Se analizaron tres configuraciones principales:

- Modelo base (BERT congelado + capa lineal)
Utilizado como baseline, donde BERT actúa como extractor de características.

- Clasificador personalizado
Se incorporó una red neuronal más profunda como cabeza de clasificación, mejorando la capacidad de modelar relaciones no lineales.

- Fine-tuning completo
Se liberaron todos los parámetros del modelo, permitiendo una adaptación total al dominio del problema.

#### Resultados obtenidos

- El modelo base mostró un desempeño moderado (~66% accuracy), evidenciando limitaciones al utilizar una capa de clasificación simple.

- El clasificador personalizado mejoró significativamente los resultados (~86%), demostrando la importancia del diseño de la capa final.

- El fine-tuning completo alcanzó un desempeño sobresaliente (~97%), posicionándose como la mejor estrategia.


### Resultados

- El modelo LSTM alcanzó una precisión aproximada del 87%, mostrando un buen desempeño general en la clasificación.

- El análisis del reporte de clasificación muestra que el modelo es mucho más preciso identificando comentarios positivos que negativos, lo cual está relacionado con el desbalance de clases presente en el dataset.

- El modelo Transformer obtuvo una precisión cercana al 83% haciendo tratamiento en el balance de clases en el conjunto de prueba, demostrando que también es capaz de aprender patrones relevantes en los textos mediante el mecanismo de atención.

### Comparación entre modelos

- Se evidencia que BERT como feature extractor es útil, pero limitado si no se adapta al problema.

- El uso de un clasificador más expresivo permite mejorar notablemente el rendimiento sin aumentar demasiado la complejidad.

- El fine-tuning completo maximiza la capacidad del modelo al ajustar sus representaciones internas al dominio específico.

Los resultados demuestran que el uso de modelos preentrenados basados en transformers, especialmente mediante fine-tuning, permite alcanzar un desempeño sobresaliente en tareas de clasificación de texto, consolidándose como el enfoque más efectivo frente a alternativas más simples.

# 9. Referencias

@article{BERTIN,
    author = {Javier De la Rosa y Eduardo G. Ponferrada y Manu Romero y Paulo Villegas y Pablo González de Prado Salas y María Grandury},
    title = {BERTIN: Efficient Pre-Training of a Spanish Language Model using Perplexity Sampling},
    journal = {Procesamiento del Lenguaje Natural},
    volume = {68},
    number = {0},
    year = {2022},
    keywords = {},
    abstract = {The pre-training of large language models usually requires massive amounts of resources, both in terms of computation and data. Frequently used web sources such as Common Crawl might contain enough noise to make this pretraining sub-optimal. In this work, we experiment with different sampling methods from the Spanish version of mC4, and present a novel data-centric technique which we name perplexity sampling that enables the pre-training of language models in roughly half the amount of steps and using one fifth of the data. The resulting models are comparable to the current state-of-the-art, and even achieve better results for certain tasks. Our work is proof of the versatility of Transformers, and paves the way for small teams to train their models on a limited budget.},
    issn = {1989-7553},
    url = {http://journal.sepln.org/sepln/ojs/ojs/index.php/pln/article/view/6403},
    pages = {13--23}
}
